# Baseline: Resolución de Ejercicios de Emparejamiento (Comprensión Lectora)

Este notebook implementa un baseline para resolver ejercicios de emparejamiento de comprensión lectora en español usando un LLM cargado con **Unsloth** para inferencia eficiente.

## Tipos de ejercicio soportados
- **Más preguntas que respuestas**: las respuestas pueden repetirse.
- **Más respuestas que preguntas**: hay distractores (respuestas que sobran).

## Flujo
1. Cargar el modelo con Unsloth
2. Construir el prompt por ejercicio
3. Realizar la inferencia
4. Parsear y estructurar las respuestas
5. Evaluar contra el ground truth

## 0. Instalación de dependencias

In [1]:
#@title Librerías necesarias
import json
import random
import torch
!pip install unsloth codecarbon
import unsloth
from unsloth import FastVisionModel
from codecarbon import EmissionsTracker
import gc
import re
import os
from google.colab import drive
from PIL import Image
from tqdm import tqdm

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
Unsloth: Your Flash Attention 2 installation seems to be broken. Using Xformers instead. No performance changes will be seen.
🦥 Unsloth Zoo will now patch everything to make training faster!


## 1. Imports y configuración

In [2]:
#@title Montar Google Drive
drive.mount('/content/drive', force_remount=True)

BASE_PATH = "/content/drive/MyDrive/MASTER/TFMs/PROFE 2025/"

DATA_PATH = os.path.join(BASE_PATH, "/data/dev/matching.json")


Mounted at /content/drive


In [3]:
# Caso 1: Más preguntas que respuestas → las respuestas pueden repetirse
SYSTEM_PROMPT_REPEATABLE = """Eres un profesor experto en resolver exámenes de comprensión lectora en español.
Tu tarea es leer atentamente el texto y emparejar cada elemento del set2 con su correspondiente opción del set1.
Las ÚNICAS opciones válidas del set1 son: {valid_ids}

IMPORTANTE: Como hay más preguntas que opciones, una misma opción del set1 puede usarse más de una vez.

Responde EXCLUSIVAMENTE basándote en la información del texto, sin utilizar conocimiento externo.

No escribas explicaciones, ni introducciones, ni texto adicional.
SOLO responde con un objeto JSON con el siguiente formato:
{{"<id_elemento_set2>": "<id_opcion_set1>", ...}}
Ejemplo de formato correcto:
{{"13": "A", "14": "C", "15": "B"}}"""


# Caso 2: Más respuestas que preguntas → hay distractores que sobran
SYSTEM_PROMPT_DISTRACTORS = """Eres un profesor experto en resolver exámenes de comprensión lectora en español.
Tu tarea es leer atentamente el texto y emparejar cada elemento del set2 con su correspondiente opción del set1.
Las ÚNICAS opciones válidas del set1 son: {valid_ids}

IMPORTANTE: Hay más opciones del set1 que elementos del set2. Algunas opciones son distractores y no deben usarse. Cada opción solo puede utilizarse una vez.

Responde EXCLUSIVAMENTE basándote en la información del texto, sin utilizar conocimiento externo.

No escribas explicaciones, ni introducciones, ni texto adicional.
SOLO responde con un objeto JSON con el siguiente formato:
{{"<id_elemento_set2>": "<id_opcion_set1>", ...}}
Ejemplo de formato correcto:
{{"13": "A", "14": "C", "15": "B"}}"""

## 2. Cargar el modelo con Unsloth

In [4]:
def load_model_unsloth(model_name, max_seq_length=2048, dtype=None, load_in_4bit=True):
    """
    Carga un modelo y su tokenizador usando Unsloth y lo prepara para inferencia.
    """
    model, tokenizer = FastVisionModel.from_pretrained(
        model_name = model_name,
        max_seq_length = max_seq_length,
        dtype = dtype,
        load_in_4bit = load_in_4bit,
    )

    FastVisionModel.for_inference(model)

    return model, tokenizer

## 3. Cargar los datos

In [5]:
def load_data():
    """Carga los ficheros JSON para evaluar los modelos."""
    with open(DATA_PATH, 'r', encoding='utf-8') as f:
        data = json.load(f)
    return data

## 4. Construcción del prompt

In [6]:
def filter_exercises(data: dict) -> list[dict]:
    """Filtra los ejercicios de emparejamiento y prepara la lista a procesar."""
    ejercicios = []
    for exam in data["exams"]:
        for ex in exam["exercises"]:
            if ex["type"] != "matching":
                continue
            ejercicios.append({
                "exerciseID"  : ex["exerciseID"],
                "level"       : exam["level"],
                "instructions": ex.get("instructions", ""),
                "exercise"    : ex["exercise"],
                "ground_truth": {
                    str(opt["optionId"]): str(opt["set1-correct-match"])
                    for opt in ex["exercise"]["set2"]
                    if "set1-correct-match" in opt
                },
            })
    return ejercicios


In [7]:
def prepare_batch(batch: list[dict]) -> tuple[list, list]:
    """
    Construye los mensajes para un batch de ejercicios de emparejamiento,
    inyectando el system prompt adecuado y formateando set1 y set2.

    Los ejercicios de emparejamiento pueden ser de dos tipos:
    - Más preguntas (set2) que opciones (set1): las respuestas se repiten.
    - Más opciones (set1) que preguntas (set2): hay distractores en set1.

    Args:
        batch:          Lista de ejercicios (cada uno con 'exerciseID',
                        'instructions' y 'exercise' con 'set1' y 'set2').
        text_template:  Plantilla de system prompt con {valid_ids} y {type_note}.

    Returns:
        mensajes_batch: Lista de conversaciones [system, user] por ejercicio.
        imagenes_batch: Lista de listas de imágenes PIL por ejercicio
                        (vacía si el ejercicio no usa imágenes).
    """
    mensajes_batch = []
    imagenes_batch = []
    text_template = None

    for ex in batch:
        exercise = ex["exercise"]
        set1     = exercise["set1"]
        set2     = exercise["set2"]

        # ── Determinar tipo de ejercicio ──────────────────────────────────────
        n_set1, n_set2 = len(set1), len(set2)
        if n_set2 > n_set1:
            text_template = SYSTEM_PROMPT_REPEATABLE
        else:
            text_template = SYSTEM_PROMPT_DISTRACTORS

        # ── Construir el system prompt ───────────────────────────────────────

        valid_ids    = ", ".join(str(opt["optionId"]) for opt in set1)
        system_prompt = text_template.format(valid_ids=valid_ids)

        # ── Construir el contenido del usuario ────────────────────────────────
        user_content  = []
        imagenes_tarea = []

        # Instrucciones del ejercicio
        instructions = ex.get("instructions", "").strip()
        user_content.append({"type": "text", "text": f"Instrucciones:\n{instructions}\n\n"})

        # ── Set1: textos/opciones ─────────────────────────────────────────────
        user_content.append({"type": "text", "text": "═══ TEXTOS / OPCIONES (set1) ═══\n"})

        for opt in set1:
            opt_id    = opt["optionId"]
            texto     = opt.get("text", "").strip()
            ruta_img  = opt.get("image-path", "")

            user_content.append({"type": "text", "text": f"[{opt_id}] "})

            if texto:
                user_content.append({"type": "text", "text": f"{texto}\n\n"})

            if ruta_img:
                ruta_absoluta = os.path.join(BASE_PATH, ruta_img)
                img_pil = Image.open(ruta_absoluta).convert("RGB")
                imagenes_tarea.append(img_pil)
                user_content.append({"type": "image"})
                user_content.append({"type": "text", "text": "\n\n"})

        # ── Set2: preguntas ───────────────────────────────────────────────────
        user_content.append({"type": "text", "text": "═══ PREGUNTAS (set2) ═══\n"})

        for opt in set2:
            opt_id   = opt["optionId"]
            texto    = opt.get("text", "").strip()
            ruta_img = opt.get("image-path", "")

            user_content.append({"type": "text", "text": f"Pregunta {opt_id}: "})

            if texto:
                user_content.append({"type": "text", "text": f"{texto}\n"})

            if ruta_img:
                ruta_absoluta = os.path.join(BASE_PATH, ruta_img)
                img_pil = Image.open(ruta_absoluta).convert("RGB")
                imagenes_tarea.append(img_pil)
                user_content.append({"type": "image"})
                user_content.append({"type": "text", "text": "\n"})

        # Prompt de respuesta
        user_content.append({"type": "text", "text": "\nRespuesta JSON:"})

        mensajes_batch.append([
            {"role": "system",  "content": system_prompt},
            {"role": "user",    "content": user_content},
        ])
        imagenes_batch.append(imagenes_tarea)

    return mensajes_batch, imagenes_batch

In [8]:
def generate_response(model, tokenizer, batch_messages, batch_imagenes, max_new_tokens):
    """Ejecuta la inferencia multimodal sobre un lote y devuelve los textos generados."""

    textos_prompt = [
        tokenizer.apply_chat_template(m, tokenize=False, add_generation_prompt=True)
        for m in batch_messages
    ]

    imagenes_planas = [img for sublista in batch_imagenes for img in sublista]

    model_inputs = tokenizer(
        text=textos_prompt,
        images=imagenes_planas if len(imagenes_planas) > 0 else None,
        padding=True,
        return_tensors="pt",
    ).to("cuda")

    with torch.no_grad():
        outputs = model.generate(
            **model_inputs,
            max_new_tokens=max_new_tokens,
            max_length=None,
            do_sample=False,
            pad_token_id=tokenizer.pad_token_id
        )

    respuestas_brutas = []
    input_lengths = model_inputs.attention_mask.sum(dim=1)  # longitud real por elemento

    for i, output in enumerate(outputs):
        gen_tokens = output[input_lengths[i]:]
        texto = tokenizer.decode(gen_tokens, skip_special_tokens=True).strip()
        respuestas_brutas.append(texto)

    return respuestas_brutas

## 6. Parseo de la respuesta

In [9]:
def process_response(texto_bruto: str, exercise: dict) -> tuple[dict[str, str], bool]:
    """
    Extrae el dict de emparejamientos desde la respuesta en bruto del modelo.

    Args:
        texto_bruto: Texto generado por el modelo (se espera un JSON).
        exercise:    Dict con 'set1' y 'set2' del ejercicio, usado para
                     validar que los IDs de la respuesta son coherentes.

    Returns:
        predicciones:  Dict {set2_optionId (str): set1_optionId (str)}.
        error_formato: True si hubo algún problema al parsear o validar.
    """
    predicciones  = {}
    error_formato = False

    valid_set1_ids    = {str(opt["optionId"]) for opt in exercise["set1"]}
    expected_set2_ids = {str(opt["optionId"]) for opt in exercise["set2"]}

    # 1. Extraer JSON de un bloque markdown si lo hay
    md_match = re.search(r"```(?:json)?\s*([\s\S]*?)```", texto_bruto)
    json_str = md_match.group(1).strip() if md_match else texto_bruto.strip()

    # 2. Si hay texto envolvente, quedarse solo con el primer objeto JSON
    if not json_str.startswith("{"):
        brace_match = re.search(r"\{[\s\S]*\}", json_str)
        json_str = brace_match.group(0) if brace_match else json_str

    # 3. Parsear
    try:
        parsed = json.loads(json_str)
    except json.JSONDecodeError:
        # Recuperación por regex: buscar pares clave-valor sueltos
        pairs = re.findall(r'["\']?(\w+)["\']?\s*:\s*["\']?(\w+)["\']?', texto_bruto)
        parsed = dict(pairs)
        error_formato = bool(not parsed)

    # 4. Normalizar y validar
    for k, v in parsed.items():
        k_str, v_str = str(k), str(v)
        if k_str not in expected_set2_ids:
            error_formato = True
            continue
        if v_str not in valid_set1_ids:
            error_formato = True
        predicciones[k_str] = v_str

    # 5. Detectar preguntas sin respuesta
    if expected_set2_ids - set(predicciones.keys()):
        error_formato = True

    return predicciones, error_formato

In [10]:
def split_tasks_by_modality(exercises: list[dict]) -> tuple[list[dict], list[dict]]:
    """Separa los ejercicios de emparejamiento en dos grupos: solo texto y con imágenes."""
    tareas_texto  = []
    tareas_imagen = []

    for ex in exercises:
        set1 = ex["exercise"].get("set1", [])
        set2 = ex["exercise"].get("set2", [])

        tiene_imagen = any(
            opt.get("image-path", "") != ""
            for opt in set1 + set2
        )

        (tareas_imagen if tiene_imagen else tareas_texto).append(ex)

    return tareas_texto, tareas_imagen

In [11]:
def run_inference(
    model,
    tokenizer,
    max_new_tokens: int = 1000,
    batch_size_texto: int = 4,
    output_file: str = "resultados.jsonl"
):
    """Ejecuta la inferencia procesando primero texto en batches y luego imágenes 1 a 1."""

    tokenizer.padding_side = "left"
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    data = load_data()
    todos_los_ejercicios = filter_exercises(data)
    tareas_texto, tareas_imagen = split_tasks_by_modality(todos_los_ejercicios)

    if os.path.exists(output_file):
        os.remove(output_file)

    def procesar_grupo(grupo_ejercicios, b_size, descripcion):
        for i in tqdm(range(0, len(grupo_ejercicios), b_size), desc=descripcion):
            batch = grupo_ejercicios[i : i + b_size]

            mensajes, imagenes = prepare_batch(batch)
            # print(mensajes)
            textos_generados   = generate_response(model, tokenizer, mensajes, imagenes, max_new_tokens)

            batch_results = []
            for j, texto_bruto in enumerate(textos_generados):
                ex = batch[j]
                predicciones, error_formato = process_response(texto_bruto, ex["exercise"])

                batch_results.append({
                    "exerciseID"              : ex["exerciseID"],
                    "level"                   : ex.get("level", ""),
                    "predicciones"            : predicciones,
                    "ground_truth"            : ex.get("ground_truth", {}),
                    "respuesta_completa"      : texto_bruto,
                    "error_procesamiento_json": error_formato,
                })

            with open(output_file, "a", encoding="utf-8") as f:
                for resultado in batch_results:
                    f.write(json.dumps(resultado, ensure_ascii=False) + "\n")

    if tareas_texto:
        print(f"\n--- Procesando {len(tareas_texto)} ejercicios de SOLO TEXTO (Batch Size: {batch_size_texto}) ---")
        procesar_grupo(tareas_texto, batch_size_texto, "Progreso Texto")

    if tareas_imagen:
        print(f"\n--- Procesando {len(tareas_imagen)} ejercicios MULTIMODALES (Batch Size: 1) ---")
        procesar_grupo(tareas_imagen, 1, "Progreso Imágenes")

    print(f"\nResultados guardados en: {output_file}")

In [12]:
model, tokenizer = load_model_unsloth("unsloth/gemma-4-E4B-it-unsloth-bnb-4bit", max_seq_length=2048, load_in_4bit=True)

==((====))==  Unsloth 2026.6.1: Fast Gemma4 patching. Transformers: 5.5.0.
   \\   /|    NVIDIA A100-SXM4-40GB. Num GPUs = 1. Max memory: 39.494 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 8.0. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors.index.json:   0%|          | 0.00/363k [00:00<?, ?B/s]

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/2130 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/203 [00:00<?, ?B/s]

processor_config.json:   0%|          | 0.00/1.69k [00:00<?, ?B/s]

chat_template.jinja:   0%|          | 0.00/16.8k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/19.9k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/32.2M [00:00<?, ?B/s]

In [15]:
gemma4_path = os.path.join(BASE_PATH, 'gemma4_results')
gemma4_results_path = os.path.join(gemma4_path, "matching_gemma4_dev.json")


In [17]:
DATA_PATH = os.path.join(BASE_PATH, "data/dev/matching.json")

In [18]:
tokenizer.padding_side = "left"
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

tracker = EmissionsTracker(
    project_name="gemma4_matching_dev",
    output_dir=gemma4_path,
    output_file="gemma4_matching_dev_emissions.csv",
    log_level="warning"
)

tracker.start()
try:
    run_inference(
        model=model,
        tokenizer=tokenizer,
        max_new_tokens=256,
        batch_size_texto=8,
        output_file=gemma4_results_path
    )
finally:
    emisiones = tracker.stop()
    print(f"Inferencia completada.")
    print(f"Emisiones estimadas: {emisiones:.4f} kg de CO2eq")

[codecarbon WARNING @ 18:24:49] Multiple instances of codecarbon are allowed to run at the same time.
[codecarbon WARNING @ 18:24:50] We saw that you have a Intel(R) Xeon(R) CPU @ 2.20GHz but we don't know it. Please contact us.
[codecarbon WARNING @ 18:24:50] No CPU tracking mode found. Falling back on estimation based on TDP for CPU. 
 Linux OS detected: Please ensure RAPL files exist, and are readable, at /sys/class/powercap/intel-rapl/subsystem to measure CPU

[codecarbon WARNING @ 18:24:50] No CPU tracking mode found. Falling back on CPU constant mode.
[codecarbon WARNING @ 18:24:50] Unable to access geographical location through primary API. Will resort to using the backup API - Exception : Region is empty - url=https://get.geojs.io/v1/ip/geo.json



--- Procesando 6 ejercicios de SOLO TEXTO (Batch Size: 8) ---


Progreso Texto: 100%|██████████| 1/1 [00:41<00:00, 41.08s/it]



--- Procesando 4 ejercicios MULTIMODALES (Batch Size: 1) ---


Progreso Imágenes: 100%|██████████| 4/4 [01:16<00:00, 19.19s/it]


Resultados guardados en: /content/drive/MyDrive/MASTER/TFMs/PROFE 2025/gemma4_results/matching_gemma4_dev.json
Inferencia completada.
Emisiones estimadas: 0.0022 kg de CO2eq


## 7. Evaluación: Accuracy sobre el ground truth

In [19]:
def compute_accuracy(output_file: str) -> dict:
    """
    Calcula el accuracy de los emparejamientos predichos frente al ground truth.

    Métricas calculadas:
    - accuracy_global   : % de emparejamientos correctos sobre el total de pares.
    - accuracy_ejercicio: % de ejercicios donde TODOS los pares son correctos.
    - por_nivel         : accuracy global desglosado por nivel (A2, B1, B2…).

    Args:
        output_file: Ruta al fichero .jsonl generado por run_inference().

    Returns:
        Dict con las métricas descritas arriba.
    """
    resultados = []
    with open(output_file, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                resultados.append(json.loads(line))

    total_pares     = 0
    correctos       = 0
    ejerc_correctos = 0
    nivel_stats     = {}  # {nivel: [total, correctos]}

    for r in resultados:
        preds = r.get("predicciones", {})
        gt    = r.get("ground_truth", {})
        nivel = r.get("level", "desconocido")

        if not gt:
            continue  # ejercicio sin ground truth, se omite

        pares_ej    = len(gt)
        aciertos_ej = sum(1 for k, v in gt.items() if preds.get(k) == v)

        total_pares += pares_ej
        correctos   += aciertos_ej

        if aciertos_ej == pares_ej:
            ejerc_correctos += 1

        if nivel not in nivel_stats:
            nivel_stats[nivel] = [0, 0]
        nivel_stats[nivel][0] += pares_ej
        nivel_stats[nivel][1] += aciertos_ej

    n_ejercicios  = len([r for r in resultados if r.get("ground_truth")])
    acc_global    = correctos / total_pares   if total_pares    > 0 else 0.0
    acc_ejercicio = ejerc_correctos / n_ejercicios if n_ejercicios > 0 else 0.0
    por_nivel     = {
        niv: round(stats[1] / stats[0], 4) if stats[0] > 0 else 0.0
        for niv, stats in sorted(nivel_stats.items())
    }

    metricas = {
        "accuracy_global"    : round(acc_global, 4),
        "accuracy_ejercicio"  : round(acc_ejercicio, 4),
        "total_pares"         : total_pares,
        "pares_correctos"     : correctos,
        "total_ejercicios"    : n_ejercicios,
        "ejercicios_perfectos": ejerc_correctos,
        "por_nivel"           : por_nivel,
    }

    # ── Impresión de resultados ──────────────────────────────────────────────
    print("=" * 50)
    print("           RESULTADOS DE EVALUACIÓN")
    print("=" * 50)
    print(f"  Accuracy global   (por par):    {acc_global:.2%}  ({correctos}/{total_pares})")
    print(f"  Accuracy ejercicio (todo OK):   {acc_ejercicio:.2%}  ({ejerc_correctos}/{n_ejercicios})")
    print()
    print("  Accuracy por nivel:")
    for niv, acc in por_nivel.items():
        stats = nivel_stats[niv]
        print(f"    {niv:<12} {acc:.2%}  ({stats[1]}/{stats[0]})")
    print("=" * 50)

    return metricas


metricas = compute_accuracy(gemma4_results_path)


           RESULTADOS DE EVALUACIÓN
  Accuracy global   (por par):    89.87%  (71/79)
  Accuracy ejercicio (todo OK):   60.00%  (6/10)

  Accuracy por nivel:
    A1           92.86%  (13/14)
    A2           100.00%  (6/6)
    B1           100.00%  (6/6)
    B2           81.08%  (30/37)
    C1           100.00%  (8/8)
    C2           100.00%  (8/8)
